# Appending nested Zarr groups with DataTree

This notebook generates daily nested Zarr v2 inputs with two spatial-resolution groups, `/0` and `/1`. `zappend` opens each input as an `xarray.DataTree` and appends all groups as one transactional slice: if one group fails, the complete daily tree is rolled back.

In [1]:
import os
import shutil
from pathlib import Path

import xarray as xr

from create_inputs import make_datatree_slices
from zappend.api import zappend

## Generate and collect nested daily inputs

Each input has a metadata-only root group and two data-bearing child groups. Both carry the same daily `time` coordinate, but `/0` is 180 by 360 pixels and `/1` is 90 by 180 pixels.

In [2]:
%%time
make_datatree_slices(31)

input_dir = Path('inputs/datatree')
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
slices = sorted(str(path) for path in input_dir.glob('*.zarr'))
assert len(slices) == 31
slices[:3]

CPU times: user 10.8 s, sys: 2.52 s, total: 13.4 s
Wall time: 10.1 s


['inputs/datatree/slice-2024-01-01.zarr',
 'inputs/datatree/slice-2024-01-02.zarr',
 'inputs/datatree/slice-2024-01-03.zarr']

## Default nested layout

The first target keeps each group's encoding from its first slice. Because the inputs use one day per time chunk, this layout is suited to maps and single-day access.

In [3]:
%%time
target = output_dir / 'datatree_example_space_opt.zarr'
count = zappend(
    slices,
    target_dir=str(target),
    force_new=True,
    logging='INFO',
)
assert count == len(slices)

2026-10-30 12:24:06,186 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-10-30 12:24:06,188 WARNING Permanently deleting outputs/datatree_example_space_opt.zarr
2026-10-30 12:24:06,274 INFO Opening slice dataset from inputs/datatree/slice-2024-01-01.zarr
2026-10-30 12:24:06,319 INFO Creating target dataset outputs/datatree_example_space_opt.zarr/0
2026-10-30 12:24:06,547 INFO Creating target dataset outputs/datatree_example_space_opt.zarr/1
2026-10-30 12:24:06,688 INFO Transaction completed.
2026-10-30 12:24:06,689 INFO Slice dataset inputs/datatree/slice-2024-01-01.zarr closed
2026-10-30 12:24:06,697 INFO Opening slice dataset from inputs/datatree/slice-2024-01-02.zarr
2026-10-30 12:24:06,743 INFO Updating target dataset outputs/datatree_example_space_opt.zarr/0
2026-10-30 12:24:07,031 INFO Updating target dataset outputs/datatree_example_space_opt.zarr/1
2026-10-30 12:24:07,148 INFO Transaction completed.
2026-10-30 12:24:07,14

CPU times: user 15.5 s, sys: 3.2 s, total: 18.7 s
Wall time: 14.9 s


In [4]:
tree = xr.open_datatree(target, engine='zarr', chunks={})
tree

<xarray.DataTree>
Group: /
│   Attributes:
│       title:        Zappend nested example
│       description:  Two spatial resolutions of daily data
├── Group: /0
│       Dimensions:       (time: 31, lat: 1800, lon: 3600)
│       Coordinates:
│         * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
│         * lat           (lat) float32 7kB -90.0 -89.9 -89.8 -89.7 ... 89.7 89.8 89.9
│         * lon           (lon) float32 14kB -180.0 -179.9 -179.8 ... 179.7 179.8 179.9
│       Data variables:
│           quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
│           temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
│       Attributes:
│           spatial_resolution:  0.1 degrees
└── Group: /1
        Dimensions:       (time: 31, lat: 900, lon: 1800)
        Coordinates:
          * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
          * lat           (lat) float32 4kB -90.0 -89.8 -89.6 -89.4 ... 89.4 89.6 89.8
          * lon           (lon) float32 7kB -180.0 -179.8 -179.6 ... 179.4 179.6 179.8
        Data variables:
            quality_flag  (time, lat, lon) uint8 50MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
            temperature   (time, lat, lon) float32 201MB dask.array<chunksize=(1, 450, 450), meta=np.ndarray>
        Attributes:
            spatial_resolution:  0.2 degrees

## Per-group time-series chunks

`group_variables` supplies optional overrides keyed by absolute group path. Here both resolutions get a full-month time chunk, while their spatial chunk shapes remain appropriate for their own grids. Any group omitted from `group_variables` would retain the shared `variables` configuration, or its first slice's encoding when no shared configuration is given.

In [8]:
%%time
full_time_chunk = len(slices)
target = output_dir / 'datatree_example_time_opt.zarr'

def variable_chunks(chunks):
    return {
        name: {'encoding': {'chunks': chunks}}
        for name in ('temperature', 'quality_flag')
    }

count = zappend(
    slices,
    target_dir=str(target),
    force_new=True,
    group_variables={
        '/0': variable_chunks([full_time_chunk, 90, 90]),
        '/1': variable_chunks([full_time_chunk, 90, 90]),
    },
)
assert count == full_time_chunk

2026-10-30 12:26:25,846 WARNING Setting 'force_new' is enabled. This will permanently delete existing targets (no rollback).
2026-10-30 12:26:25,847 WARNING Permanently deleting outputs/datatree_example_time_opt.zarr
2026-10-30 12:26:25,987 INFO Opening slice dataset from inputs/datatree/slice-2024-01-01.zarr
2026-10-30 12:26:26,048 INFO Creating target dataset outputs/datatree_example_time_opt.zarr/0
2026-10-30 12:26:30,854 INFO Creating target dataset outputs/datatree_example_time_opt.zarr/1
2026-10-30 12:26:32,105 INFO Transaction completed.
2026-10-30 12:26:32,106 INFO Slice dataset inputs/datatree/slice-2024-01-01.zarr closed
2026-10-30 12:26:32,113 INFO Opening slice dataset from inputs/datatree/slice-2024-01-02.zarr
2026-10-30 12:26:32,166 INFO Updating target dataset outputs/datatree_example_time_opt.zarr/0
2026-10-30 12:26:39,677 INFO Updating target dataset outputs/datatree_example_time_opt.zarr/1
2026-10-30 12:26:41,479 INFO Transaction completed.
2026-10-30 12:26:41,480 INF

CPU times: user 3min 26s, sys: 49.4 s, total: 4min 16s
Wall time: 2min 45s


In [9]:
tree = xr.open_datatree(target, engine='zarr', chunks={})
tree

<xarray.DataTree>
Group: /
│   Attributes:
│       title:        Zappend nested example
│       description:  Two spatial resolutions of daily data
├── Group: /0
│       Dimensions:       (time: 31, lat: 1800, lon: 3600)
│       Coordinates:
│         * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
│         * lat           (lat) float32 7kB -90.0 -89.9 -89.8 -89.7 ... 89.7 89.8 89.9
│         * lon           (lon) float32 14kB -180.0 -179.9 -179.8 ... 179.7 179.8 179.9
│       Data variables:
│           quality_flag  (time, lat, lon) uint8 201MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
│           temperature   (time, lat, lon) float32 804MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
│       Attributes:
│           spatial_resolution:  0.1 degrees
└── Group: /1
        Dimensions:       (time: 31, lat: 900, lon: 1800)
        Coordinates:
          * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
          * lat           (lat) float32 4kB -90.0 -89.8 -89.6 -89.4 ... 89.4 89.6 89.8
          * lon           (lon) float32 7kB -180.0 -179.8 -179.6 ... 179.4 179.6 179.8
        Data variables:
            quality_flag  (time, lat, lon) uint8 50MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
            temperature   (time, lat, lon) float32 201MB dask.array<chunksize=(31, 90, 90), meta=np.ndarray>
        Attributes:
            spatial_resolution:  0.2 degrees

In [10]:
tree.close()
if os.path.exists(input_dir):
    shutil.rmtree(input_dir)